# J'ai deux passions : la carto, la manifestation

## Introduction

Les manifestations du 17 octobre ont été d'une ampleur rare en France. L'anticipation de cet évènement était telle que certain&middot;es ont crié à la "fin de l'époque contemporaine". On sait depuis Siegfried (1913) que d'un territoire à l'autre, l'expression politique change du tout au tout, et que les facteurs géographiques sont déterminants dans la politique française.

Une très grande hypothèse de ce projet est la suivante : les manifestant&middot;es, dans leur grande bonté, ont mis à disposition une cartographie exhaustive des évènements du 17 octobre. On suppose que tout évènement déclaré a bien eu lieu ; en retour, tout évènement qui a eu lieu a bien été déclaré. 

Ce mouvement, qui revêtait au départ un aspect très flou, s'est doté de structures et d'organisation pour permettre une mobilisation coordonnée (entre les participant&middot;es, et non par un organe central). Cela fait peut-être suite au mouvement des Gilets Jaunes : la carte que nous avons étudiée permettait d'émettre un signal crédible de la présence de militant&middot;es.

Le but de ce projet est double. 
- D'une part, il s'agit d'obtenir un aperçu, quoique partiel, des déterminants géographiques des évènements du 17 octobre. Quelles sont les prédicteurs, au niveau local, de la présence d'un évènement ? 
- De l'autre, il s'agit de se demander l'échelle la plus pertinente pour analyser ces phénomènes, dans la continuité des interrogations en géographie économique sur le *Modifiable Aeral Unit Problem* (MAUP). Et si l'échelle retenue affectait grandement la performance des prédicteurs trouvés auparavant ? 

Ce document s'organise de la manière suivante. Après avoir donné un aperçu des données, nous donnons les statistiques descriptives sous formes de cartes et histogramme des types et emplacements des actions militantes. Ensuite, nous cherchons à identifier les déterminants socio-démographiques qui expliquent pourquoi certaines communes ont connu des mobilisations le 17 octobre contrairement à d'autres. Enfin, nous nous interrogeons l'effet de l'échelle géographique choisie par le/la statisticien&middot;nes sur la présentation des mobilisations.     

Importation des modules:

In [ ]:
# ceci permet de télécharger les modules qui seront utilisés tout au long du projet
%pip install -r requirements.txt

In [42]:
# nous importons les premiers modules qui vont être utilisés pour le traitement des données 
%reload_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import folium
import useful.useful_db as udb
import ipywidgets

##  Comment avons-nous webscrapé la carte du 17 octobre ? 

### Le point de départ

 Nous sommes partis de la carte interactive du site https://17octobregiletjaune.net/carte/ qui répertorie les actions du 17 octobre. Les données sur les actions, y compris le type (manifestation, assemblée, blockage etc.) et leur coordonnées géographiques, sont téléchargées par le JavaScript de la page (une requête vers /api/actions), utilisées pour afficher des "pop-ups", puis nettoyées. 

### La méthode 

Il s'agit de figer le code au bon moment avec des points d'arrêt/ breakpoints. On a cherché les endroits où les données viennent d'être téléchargées ou traitées. Le déclic vient de la fonction qui crée les marqueurs de la carte : Une fois lieux et actions dans la console, il suffit de les copier et les coller dans un fichier csv. 

Vous pouvez télécharger le tuto complet dans [ce fichier](bdd/explication_scraping.pdf).


Importation des bases :

In [43]:
# table qu'on a scrapée
from bdd.AR_S3 import recuperer_depuis_s3

recuperer_depuis_s3("just17.csv", "/home/onyxia/work/python_manifestation_explosion/bdd/just17.csv")

# table d'appartenance des communes à des zones géographiques
recuperer_depuis_s3("appartenance_commune.xlsx", "/home/onyxia/work/python_manifestation_explosion/bdd/appartenance_commune.xlsx")

'/home/onyxia/work/python_manifestation_explosion/bdd/appartenance_commune.xlsx'

Au format DataFrame :

In [44]:
# ouverture du dataframe qui contient toutes les communes françaises pour pouvoir ensuite comparer les communes qui
# ont eu un événement par rapport aux autres 
scraped = pd.read_csv("/home/onyxia/work/python_manifestation_explosion/bdd/just17.csv", sep=";")

all_communes = pd.read_excel(
    "/home/onyxia/work/python_manifestation_explosion/bdd/appartenance_commune.xlsx",
    header=5,
    dtype=str,
    engine="calamine",
)


## Aperçu des données

### Valeurs manquantes

Nous avons scrapé ce [site](https://www.17octobregiletjaune.net/carte/17-octobre/), créé par Benjamin Ball, qui permettait à chacun&middot;e de renseigner un point de mobilisation pour le 17 octobre, et de se déclarer comme participant&middot;e. Ces mobilisations sont renseignées dans le fichier `scraped.csv`. Un aperçu rapide de la table donne :

In [ ]:
scraped.head(5)
print(f"Cette table contient {len(scraped)} lignes, correspondant à autant d'évènements.")
print("Ses colonnes sont : ",scraped.columns.to_list())


On enlève les sites pour lesquels on n'a pas de coordonnées, qui ne pourront de toute façon pas être utilisées

In [45]:
scraped_cleaned = scraped.dropna(subset=["lat","lon"])
print(f"Il y a {len(scraped_cleaned)} lignes où sont renseignées des coordonnées géographiques.")

scraped_dirty = scraped[scraped["lat"].isna()|scraped["lon"].isna()]
print(f"Il y a {len(scraped_dirty)} lignes où ne sont pas renseignées de coordonnées géographiques.")

Il y a 2009 lignes où sont renseignées des coordonnées géographiques.
Il y a 228 lignes où ne sont pas renseignées de coordonnées géographiques.


Penchons nous de plus près sur ces lignes sans coordonnées :

In [ ]:
nb_villes = scraped_dirty["ville"].notna().sum()
print(f"Il y a {nb_villes} lignes où la ville est renseignée, sans coordonnées géographiques.")

Elles nous inquiètent moins désormais : on peut les enlever en toute sécurité.

In [46]:
scraped = scraped_cleaned

## I. Statistiques et cartes descriptives

Sur le site, les participant&middot;es étaient invité&middot;es à renseigner le type d'évènement qui allait se dérouler. Ces types sont renseignés dans la colonne `title` :

In [ ]:
types_events = scraped["title"].unique().tolist()
types_events_freq = scraped["title"].value_counts(dropna=False)
display(types_events_freq)

NB: une "opération escargot" correspond à un ralentissement volontaire de la circulation par des automobilistes coordonné&middot;es. Un "piquet de grève" est une rassemblement de grèvistes et/ou manifestan·es aux abords d'un lieu de travail ou d'études pour perturber l'activité et inciter à rejoindre la grève.  

En termes de distribution dans l'espace, voici ce qui apparait :

In [47]:
carte_all_events = udb.carte_events(scraped)
carte_all_events

On notera qu'on ne pouvait pas reprocher à celleux qui ont tenté de prendre l'Elysée de n'avoir pas déclaré leur action (sous la bannière "Autre" : tout aurait été possible !).

Afin de faire des statistiques sur la distribution des évènements en France, on y ajoute le code Insee pour chacune des communes. Pour cela, on essaie de croiser directement avec un fond de carte fourni par etalab à [cette adresse](https://etalab-datasets.geo.data.gouv.fr/contours-administratifs/) :

In [ ]:
scraped_insee = udb.ajouter_code_insee(scraped)
print(f"On n'a pas de perte : il y a {len(scraped_insee) - len(scraped)} lignes différentes entre `scraped` et `scraped_insee`")

On la croise ensuite avec une table avec toutes les communes françaises, fournie par l'Insee.

In [ ]:
all_communes_events = udb.merge_communes(scraped_insee, all_communes)
# ceci permet d'enregistrer cette base de données dans le s3
# from bdd.AR_S3 import envoyer_vers_s3
# all_communes_events.to_csv(
#    "/home/onyxia/work/python_manifestation_explosion/bdd/all_communes_events.csv",
#    sep=";",
#    index=False,
#)
# envoyer_vers_s3("/home/onyxia/work/python_manifestation_explosion/bdd/all_communes_events.csv")

Cela nous permet de faire des statistiques à l'échelle de la France. 

In [ ]:
udb.desc_types_events(all_communes_events)

Lecture : 1,7% des communes ont connu des manifestations. 

In [ ]:
udb.desc_bool(all_communes_events) 

### Descriptions des actions par départements et par communes 
Dans cette partie nous allons générer des cartes qui renvoient un message sur les zones de France où les événements sont les plus importants. Nous commençons par une carte par département où le dégradé de couleur de blanc à rouge suit le nombre de déclaration d'évenements. Cette carte est interactive. La carte suivante est statique.  

In [ ]:
import sys
sys.path.append("/home/onyxia/work/python_manifestation_explosion/useful")
import chloroplethe_stat_des as chp

# Carte réactive
chp.carte_choroplethe_departements(scraped)

In [ ]:
import sys
sys.path.append("/home/onyxia/work/python_manifestation_explosion/useful")
import chloroplethe_stat_des as chp

# Version statique
chp.carte_choroplethe_statique(scraped)

Comme le département est une unité qui peut donner une impression faussée, nous reproduisons la carte par commune. On remarque qu'effectivement, ce sont les communes les plus importantes en nombre d'habitants qui font que les départements sont de plus en plus rouges.

In [ ]:
from carte_colore_commune import carte_choroplethe_interactive

m = carte_choroplethe_interactive(
    scraped,
    niveau="commune",
    uniquement_actifs=True,
)
m

Nous remarquons que Paris rassemble 83 actions et donc écrase les couleurs des autres villes. Nous allons donc essayer de contrôler ce fait pour isoler d'autres impacts sociodémographiques en pondérant la colorimétrie par le nombre d'habitants par commune. 

In [ ]:
from bdd.AR_S3 import recuperer_depuis_s3
recuperer_depuis_s3("commune_events_pop.csv","/home/onyxia/work/python_manifestation_explosion/bdd/commune_events_pop.csv")
commune_events_pop = pd.read_csv("/home/onyxia/work/python_manifestation_explosion/bdd/commune_events_pop.csv", sep=";")

from carte_colore_commune import preparer_table_ponderee,carte_statique_ponderee
m = carte_statique_ponderee(commune_events_pop,col_pop="PMUN",pop_min=1000)
m

Nous voyons sur cette carte que Paris devient moins rouge du fait du nombre d'habitants qui est très important. Cependant, cela ne veut pas dire que peu de gens ont manifesté car une action peut rassembler plusieurs dizaines de milliers de personne. Nous ne pouvons pas vraiment contrôler ce type de biais. Nous allons tout de même faire en sorte de voir si les types d'action varient en fonction de la taille de la ville. Pour ce faire, nous construisons l'histogramme suivant. 

In [ ]:
from useful.histogramme import plot_genre_par_type_commune
ax = plot_genre_par_type_commune(commune_events_pop, normalize="columns")
ax

Lecture : à nombre d'actions donné dans les très grandes villes ce sont 40% des actions qui sont des manifestations et environ 10% des assemblées. 

## II. Impact des variables socio-démographiques

Pour comprendre si les actions du 17 octobre se sont tenues dans des communes avec des caractéristiques socio-démographiques particulières, nous avons d'abord regardé : 

- la **taille** de la commune (nombre d'inscrits, diffère légèrement du nombre d'habitants) ;
- son **niveau d'équipement** : nombre d'équipements de la Base permanente des équipements (Insee) pour 1 000 habitants ;
- son **vote** au 1er tour des législatives 2024 : part de chaque bloc politique et participation.


Les données sont tirées des fichiers suivants (tous disponibles sur le s3)
- `commune_events_pop.csv`, une ligne par commune, avec `has_event` = la commune a-t-elle accueilli un événement ? 
-  `bpe.csv`, Base permanente des équipements (Insee), un équipement par ligne
- `vote.csv`, les résultats du 1er tour des législatives 2024 par commune.  
  
*Remarque*: Les nuances politiques sont regroupées en cinq blocs : extrême gauche, gauche, droite, extrême droite, divers (qui semble renvoyer avant tout aux candidats régionalistes). On raisonne par blocs pour plus de clarté et car l'offre électorale change d'une circonscription à l'autre
(l'extrême droite est représentée tantôt par un candidat RN, tantôt UXD).

Tout le traitement est dans `explications/poisson.py`. 

In [48]:
# ceci permet d'obtenir les résultats suivants
%run explications/poisson.py


Table : 34,860 communes | 1,124 avec un événement (3.2%)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLEU, ORANGE, GRIS = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e8e7e3", "axes.axisbelow": True})

# Classes de taille (en nombre d'inscrits)
bornes = [0, 100, 500, 2000, 10000, 30000, 50000, 100000, np.inf]
noms = ["< 100", "100–499", "500–1 999", "2 000–9 999", "10 000–29 999",
        "30 000–49 999", "50 000–99 999", "100 000 et +"]
df["classe_taille"] = pd.cut(df["Inscrits"], bornes, labels=noms, right=False)

par_taille = df.groupby("classe_taille", observed=True)["a_eu_action"].agg(["mean", "size"])

fig, ax = plt.subplots(figsize=(11, 4.5))
barres = ax.bar(par_taille.index.astype(str), par_taille["mean"] * 100, color=BLEU, width=0.6)
for b, (taux, n) in zip(barres, par_taille.itertuples(index=False)):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1, f"{taux:.0%}\n({n:,} communes)".replace(",", " "),
            ha="center", va="bottom", fontsize=9)
ax.tick_params(axis ="x", rotation = 20)
ax.set_ylabel("Communes avec un événement (%)")
ax.set_xlabel("Nombre d'inscrits")
ax.set_ylim(0, min(100, par_taille["mean"].max() * 100 * 1.3))
ax.set_title("Part des communes ayant accueilli un événement, selon leur taille", loc="left")
plt.tight_layout()
plt.show()

La probabilité d'accueillir un événement augmente très fortement avec la taille de la commune.
Or la taille est aussi liée au vote (les grandes communes votent davantage à gauche, les petites
davantage à l'extrême droite et participent plus) et aux équipements présents. **Toute comparaison entre communes doit donc
se faire à taille égale**, sinon on attribue au vote ou aux mauvais/bon équipement ce qui relève de la taille.

**Le modèle de régression**

On explique le fait d'avoir accueilli un événement (0 ou 1) par une régression de Poisson
avec erreurs-types robustes, groupées par département (« Poisson modifié »). Cette méthode donne
directement des **rapports de probabilités** :

$$\text{effet en \%} = \left(e^{\hat\beta} - 1\right) \times 100$$

se lit « la probabilité d'avoir accueilli un événement est X % plus élevée ».

**Lecture des variables.**
- Vote et participation : effet de **+10 points**. Les parts des blocs somment à 100 %, la **droite** sert de
  référence.
- Équipement : effet d'**un écart-type** de plus (log des équipements pour 1 000 inscrits).
- Taille : `log_inscrits`, effet pour une commune 2,7 fois plus grande.

**Trois modèles**, de plus en plus exigeants :
(1) vote + taille ; (2) + participation et équipement ; (3) + effets fixes département,
c'est-à-dire qu'on ne compare que des communes **d'un même département**.

In [ ]:
print("\n" + "=" * 90)
print("Variation de la probabilité d'avoir eu un événement (en %)")
print("  vote et participation : pour +10 points | équipement : pour +1 écart-type")
print("  log_inscrits : pour une commune 2,7 fois plus grande | bloc de référence : droite")
print("=" * 90)
print(effets.round(1).to_string())
print("\np-values (< 0,05 : effet significatif au seuil de 5 %)")
print(pvalues.round(3).to_string())
print(f"\nModèle (3) : {(~dep_avec_evenement).sum():,} communes retirées "
      f"(départements sans aucun événement)")


**Quelques résultats**

- **La taille** : plus une commune est grande, plus elle a de chances d'avoir accueilli un événement
- **L'équipement** : A taille égale et au sein du même département, une commune nettement mieux équipée (écart-type +1) a environ 2 fois plus de chances d'avoir connu au moins un événement le 17 octobre. Cela semble cohérent avec le fait que les événements se tiennent dans les pôles locaux, mieux équipés en termes de commerces, écoles, hôpitaux etc.,  qui rassemblent aussi les habitants des communes voisines. 
- **Le vote** : Le lien entre vote à gauche et événement disparaît dès qu'on contrôle la taille de la commune. Les effets extrême gauche et bloc divers disparaissent également lorsqu'on se place à l'échelle d'un même département (pas significatifs). Le vote à l'extrême droite au sein d'une commune est corrélé positivement au fait qu'une action ait eu lieu (un résultat qu'on ne s'explique pas). Autre effet surprenant : une participation électorale plus forte au sein d'une commune s'accompagne d'une probabilité d'événement plus faible. Il faudrait sans doute contrôler par l'âge car les personnes âgées sont celles qui votent le plus et potentiellement qui se mobilisent moins dans un mouvement lycéen. 

Limite : Ce sont des **corrélations** entre communes qui ne disent rien de l'effet causal, ni de qui a participé aux événements.

## III. Est-ce la taille, ou la forme qui compte ? Exploration du MAUP

Le MAUP est un problème qui se pose en premier lieu à celui ou celle qui tente de donner des explications aux phénomènes géographiques. Il est prouvé depuis que l'échelle de l'analyse change les résultats, et ce de deux manières (Briant et al., 2010) : 
1. l'agglomération des zones, c'est-à-dire leur regroupement en des ensembles plus ou moins grans, change la donne (de la commune, au département, à la région)
2. la forme des zones, c'est-à-dire le découpage de ces zones (commune VS zone d'emploi)

Schématiquement, cela donne : 

![Fig 1 : Briant et al., 2010](modele_echelle/MAUP_simple.png)

Nous allons ici essayer de comparer les performances des modèles prédictifs, en fonction des découpages géographiques choisis, en se focalisant sur la taille de la zone (premier effet). Nous faisons ce choix car il est difficile de trouver des découpages différents dans leur forme mais relativement égaux en termes de population.

Pour chaque niveau d'agrégation (par exemple commune, département, région) et chaque unité $i$, on note $Y_i$ le nombre de manifestations (`nb_events`), $P_i$ la population et $x_i$ le prix moyen de l'essence, en centimes. Comme $Y_i$ est un comptage, on estime une régression de Poisson avec lien logarithmique, selon ce modèle :


$$\ln \mathbb{E}[Y_i \mid x_i, P_i] = \beta_0 + \beta_1 x_i + \gamma \ln P_i$$

Ce modèle est particulièrement adapté au MAUP, car le paramètre $\gamma$ renseigne directement sur la façon dont le nombre de manifestations évolue avec la taille des unités. Si la proportionnalité à la population ($\gamma = 1$) tient à une échelle mais pas à une autre, le découpage affecte l'interpétation. On compare donc, pour chaque niveau d'agrégation, $\hat\beta_1$ (via $e^{\hat\beta_1}$), $\hat\gamma$ et le test de $H_0 : \gamma = 1$. La stabilité de $\hat\beta_1$ entre échelles et entre spécifications est notre indicateur de robustesse au MAUP.

L'estimation se fait par pseudo-maximum de vraisemblance de Poisson, convergent dès que la moyenne conditionnelle est bien spécifiée, avec des écarts-types robustes (HC3). Pour tester l'**effet de zonage**, on peut en outre répéter l'estimation sur des découpages alternatifs de même échelle (par exemple des regroupements aléatoires de communes en unités de taille comparable) et observer la dispersion de $\hat\beta_1$.

**Limites.** Les coefficients sont des associations conditionnelles et non des effets causaux (que sont dans les cours des prochaines semaines en économétrie). Les écarts-types ne sont pas directement comparables d'une échelle à l'autre, puisque le nombre d'unités change (et avec lui la puissance des tests). Enfin, l'indépendance entre unités est une hypothèse discutable, d'autant plus que les unités sont petites et voisines.

In [33]:
import modele_echelle.preprocessing as pp
import statsmodels.formula.api as smf
import statsmodels.formula as sm

### Préparation des données

Il nous faut non seulement les prix des carburants dans les villes avec évènement, mais aussi sans. Pour cela, on récupère cette [base de données](https://donnees.roulez-eco.fr/opendata/annee/2026), qui contient tous les prix de toutes les stations essence françaises en 2026. Attention, cliquer sur le lien lance le téléchargement de la base de données. On ne peut plus accéder au jour précis car nous faisons ce travail plus d'un mois après le 17 octobre (preuve s'il en faut qu'il aurait mieux valu faire ce travail en avance). Nous sélectionnons le jour qui nous intéresse, le 17 octobre, dans le fichier `modele_echelle/base_carburants.py`. Il nous faut maintenant faire la jointure, en faisant une moyenne du prix de l'essence dans les villes pour lesquelles on n'a pas d'évènement (on va d'ailleurs remplacer les prix indiqués dans notre scraping, pour inclure toutes les essences, et non seulement le e10 et le gazole). Cette jointure est faite à l'aide d'une table de correspondances entre les codes Insee et les codes postaux (utilisés dans la base de données carburants).

In [34]:
import modele_echelle.base_carburants as bc

prix_17_10 = pd.read_csv("bdd/prix_2026-10-17.csv")
communes_events_carburants = bc.merge_carburants_communes(all_communes_events, prix_17_10)

### Par taille de la zone

Intéressons-nous d'abord à la taille de la zone retenue. On peut ici faire des `group_by` à différentes échelles, pour voir les effets des agglomérations. On ajoute la population. Pour les variables, on utilise une moyenne pondérée par la population du prix de l'essence dans la zone (si une commune est très  peuplée, le prix de son essence vaudra plus dans la moyenne, relativement aux autres communes de la zone). 

On le fait donc au niveau du département, en préparant d'abord les tables de notre échantillon :

In [36]:
communes_pop = pd.read_csv("bdd/pop_commune.csv", sep=";")[["COM","PMUN"]]
communes_events_carburants_pop = pd.merge(
    right=communes_pop,
    left=communes_events_carburants,
    left_on="CODGEO",
    right_on="COM",
    how="inner"
)

table_dep = pp.table_reg_essence_prix(communes_events_carburants_pop, "DEP")
display(table_dep.sample(10))

,DEP,pop_totale,prix_moyen,evenements
84,85,713609,222.160597,15
70,70,233185,226.334296,7
3,04,168054,222.269337,8
51,51,563076,224.366692,18
22,24,417614,223.866817,31
43,43,228654,225.824864,8
36,36,216069,224.439691,11
1,02,523342,223.566749,23
78,79,375229,222.000778,9
57,57,1051309,228.761170,28


Il semble que l'on n'ait pas de données renseignées sur le site quant aux prix de l'essence dans les DROM COM. En effet, en sachant que les codes des départements d'Outre mer sont tous du format "97_" : 

In [ ]:
display(table_dep[table_dep["prix_moyen"].isna()])

On peut donc `dropna`, en faisant une croix sur les Outre-Mer (comme une impression de déjà vu).

In [37]:
table_dep=table_dep.dropna()

Maintenant, pour la région :

In [38]:
table_region = pp.table_reg_essence_prix(commune_events_pop, 'REG').dropna()
display(table_region.sort_values('REG', ascending=False))

,REG,pop_totale,prix_moyen,evenements
17,94,355486.0,228.486180,10
16,93,5218960.0,222.757160,161
15,84,8205557.0,227.883063,305
14,76,6124653.0,227.082797,236
13,75,6150451.0,221.422458,209
11,53,3449370.0,221.606929,99
10,52,3907156.0,224.742675,130
9,44,5563378.0,230.917605,136
7,32,5992194.0,223.816784,163
5,28,3345842.0,224.050709,91


On prépare, notamment pour la population :

In [39]:
table_region_clean = table_region.copy()
table_dep_clean = table_dep.copy()
# Calcul explicite des logarithmes des populations
table_region_clean["log_pop"] = np.log(table_region_clean["pop_totale"])
table_dep_clean["log_pop"] = np.log(table_dep_clean["pop_totale"])

On fait maintenant des régressions en utilisant `statsmodels` :
- Pour les départements

In [40]:
mod_dep = smf.poisson(
    "evenements ~ prix_moyen + log_pop",
    data=table_dep_clean,
).fit(cov_type="HC1")

print(mod_dep.summary())

Optimization terminated successfully.
         Current function value: 4.008154
         Iterations 6
                          Poisson Regression Results                          
Dep. Variable:             evenements   No. Observations:                   95
Model:                        Poisson   Df Residuals:                       92
Method:                           MLE   Df Model:                            2
Date:                Tue, 24 Nov 2026   Pseudo R-squ.:                  0.3884
Time:                        22:05:29   Log-Likelihood:                -380.77
converged:                       True   LL-Null:                       -622.56
Covariance Type:                  HC1   LLR p-value:                9.869e-106
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.5396      3.001     -0.180      0.857      -6.421       5.342
prix_moyen    -0.0253      0.

La p-valeur à $0.86$ pour l'intercept importe peu : on s'intéresse à l'effet des variables.

Pour les régions :

In [41]:
mod_region = smf.poisson(
    "evenements ~ prix_moyen + log_pop",
    data=table_region_clean,
).fit(cov_type="HC1")

print(mod_region.summary())

Optimization terminated successfully.
         Current function value: 6.360369
         Iterations 6
                          Poisson Regression Results                          
Dep. Variable:             evenements   No. Observations:                   13
Model:                        Poisson   Df Residuals:                       10
Method:                           MLE   Df Model:                            2
Date:                Tue, 24 Nov 2026   Pseudo R-squ.:                  0.7385
Time:                        22:05:33   Log-Likelihood:                -82.685
converged:                       True   LL-Null:                       -316.16
Covariance Type:                  HC1   LLR p-value:                4.024e-102
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -12.5603      5.965     -2.106      0.035     -24.252      -0.869
prix_moyen     0.0212      0.

Ici, la p-value est très haute, effet du très petit nombre d'observations (de plus de 90, on passe à 13).

Mais le MAUP reste bien illustré : le signe devant le prix moyen a changé ! Le lissage de la variance intergroupe a pour conséquences une interprétation perturbée.

## Conclusion 

Ce travail nous a permis de répondre partiellement aux interrogations que nous avions formulées. Même si la robustesse toute relative de nos résultats ne permet pas d'interprétations causales, il nous a semblé que ces données nouvelles méritaient que l'on s'en empare. 

On peut y voir plus clair sur la répartition spatiale du 17 octobre. Cependant, une cartographie complète de la mobilisation en elle-même demanderait une incursion sur les réseaux sociaux (voir Boyer et al. 2020), et une compréhension plus fine des revendications autour desquelles s'est structuré le mouvement. Pas sûr&middot;e que la cherté de l'essence explique à elle seule la volonté d'en finir avec l'époque contemporaine. Il faut aussi dire que déclaration ne vaut pas présence : les données ne permettent pas de déduire le nombre de participant&middot;es à chaque évènement, information clé pour comprendre la dynamique de cette journée.

Notre troisième partie nous alerte sur l'importance du travail de cadrage que doit effectuer le/a *data scientist* : les données peuvent parler, mais ce qu'elles disent dépend grandement de leur traitement. Présenter des résultats, qui plus à caractère si politique, demande donc de s'assurer de la fiabilité de ses analyses, mais exige surtout une honnêteté sur ce que l'on peut en dire. Nous avouons donc en toute humilité que nos choix et le caractère partiel de ce travail appellent des analyses plus poussées de ces données, plus qu'elles n'en épuisent les possibilités.

Nous sommes reconnaissant&middot;es envers Benjamin Ball, sans qui le projet n'aurait pas vu le jour ; tout notre soutien dans son combat. 

### Références

Boyer, Pierre C., Thomas Delemotte, Germain Gauthier, Vincent Rollet, et Benoît Schmutz. 2020. « The origins of the gilets jaunes movement ». Revue économique Vol. 71 (1): 109‑38. https://doi.org/10.3917/reco.711.0109.
Briant, A., P. P. Combes, et M. Lafourcade. 2010. « Dots to Boxes: Do the Size and Shape of Spatial Units Jeopardize Economic Geography Estimations? » Journal of Urban Economics 67 (3): 287‑302. https://doi.org/10.1016/j.jue.2009.09.014.
Siegfried, André, et Christian Vandermotten. (1913) 2010. Tableau politique de la France de l’Ouest sous la Troisième République. UBlire 11. Nouvelle éd. Éd. de l’Université de Bruxelles diff. Sodis diff. ToThèmes.
